In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
# Thư viện quan trọng cho thuật toán mới
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

In [2]:
# ===============================#
#      HÀM ĐÁNH GIÁ MÔ HÌNH      #
# ===============================#
# (Giống hàm bạn đã dùng trong file 03b_GaussianNB)
def evaluate_model(y_true, y_pred, model_name, dataset_name):
    print(f"\n{'='*60}")
    print(f"{model_name} - {dataset_name}")
    print(f"{'='*60}")

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"- Accuracy:  {acc:.4f}")
    print(f"- Precision: {prec:.4f}")
    print(f"- Recall:    {rec:.4f}")
    print(f"- F1-Score:  {f1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Không bị', 'Bị']))

In [3]:
# ===========================
# ĐỌC DỮ LIỆU (MEAN hoặc MEDIAN)
# ===========================
# Random Forest không cần dữ liệu chuẩn hóa, nên dùng file _median là tốt
try:
    df2 = pd.read_csv("../../datasets/processed/diabetes_dataset2_median.csv")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file. Hãy chắc chắn đường dẫn `../../datasets/processed/diabetes_dataset2_median.csv` là chính xác.")
    # Xử lý lỗi nếu cần
    exit()

X = df2.drop('Outcome', axis=1)
y = df2['Outcome']

# =G=========================
# CHIA TRAIN/TEST (Như cũ)
# ===========================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# ========================================================
# BƯỚC MỚI: TRAIN RANDOM FOREST VỚI CLASS_WEIGHT
# ========================================================
print("\nĐang huấn luyện Random Forest với class_weight='balanced'...")

# Khởi tạo mô hình
# class_weight='balanced' sẽ tự động "phạt" nặng hơn khi dự đoán sai lớp '1'
rf_model = RandomForestClassifier(
    n_estimators=100,         # 100 cây quyết định (có thể tinh chỉnh)
    class_weight='balanced',  # Tham số quan trọng để cải thiện Recall
    random_state=42
)

# Huấn luyện trên tập train GỐC
rf_model.fit(X_train, y_train)

# Đánh giá trên tập TEST GỐC
y_pred = rf_model.predict(X_test)

# In kết quả
evaluate_model(y_test, y_pred, "Random Forest (Class Weight)", "Dataset 2 (Median)")


Đang huấn luyện Random Forest với class_weight='balanced'...

Random Forest (Class Weight) - Dataset 2 (Median)
- Accuracy:  0.7597
- Precision: 0.6607
- Recall:    0.6727
- F1-Score:  0.6667

Confusion Matrix:
[[80 19]
 [18 37]]

Classification Report:
              precision    recall  f1-score   support

    Không bị       0.82      0.81      0.81        99
          Bị       0.66      0.67      0.67        55

    accuracy                           0.76       154
   macro avg       0.74      0.74      0.74       154
weighted avg       0.76      0.76      0.76       154



In [ ]:
import pandas as pd
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- CẤU HÌNH ---
DATA_PATH = '../../datasets/processed/diabetes_dataset2_median.csv' 
MODEL_PATH = '../../models/diabetes_rf_full.pkl' 

# 1. Load dữ liệu
print(f"⏳ Đang đọc dữ liệu từ {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)

# Tách Features (X) và Target (y)
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

print(f"✅ Đã load {len(df)} dòng dữ liệu.")

# ---------------------------------------------------------
# PHẦN 1: TEST THỬ (Để xem chất lượng trước khi train full)
# ---------------------------------------------------------
print("\n--- 📊 BẮT ĐẦU TEST ĐÁNH GIÁ (Split 80/20) ---")

# Chia tạm 20% để test
X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Cấu hình model (RandomForest với class_weight balanced để tối ưu Recall)
rf_temp = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_temp.fit(X_train_temp, y_train_temp)

# Dự đoán thử
y_pred_temp = rf_temp.predict(X_test_temp)

# In kết quả
print("Confusion Matrix (Ma trận nhầm lẫn):")
print(confusion_matrix(y_test_temp, y_pred_temp))
print("\nBáo cáo chi tiết:")
print(classification_report(y_test_temp, y_pred_temp))
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_temp, y_pred_temp):.4f}")

# ---------------------------------------------------------
# PHẦN 2: TRAIN FULL VÀ LƯU (Dùng cho ứng dụng thật)
# ---------------------------------------------------------
print("\n--- 🚀 BẮT ĐẦU TRAIN TRÊN TOÀN BỘ DỮ LIỆU (FULL DATA) ---")
print("Đang huấn luyện lại trên 100% dữ liệu... Vui lòng đợi...")

# Khởi tạo lại model mới tinh
final_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# Fit trên toàn bộ X và y (Không chia cắt nữa)
final_model.fit(X, y)

# Lưu model
joblib.dump(final_model, MODEL_PATH)

print(f"\n✅ HOÀN TẤT! Model đã được lưu tại: {MODEL_PATH}")
print("Bạn có thể dùng file này cho Backend/Web App ngay.")